## 2 Notebook - Camada Bronze

Crie um notebook no Databricks responsável por:

### 1. Criar o banco de dados (database) da camada Bronze

### 2. Criar tabelas na camada bronze a partir de cada CSV

Em cada uma dessas tabelas, deve haver uma coluna chamada `timestamp_ingestion`, que deverá constar o timestamp do momento exato em que o dado foi inserido na camada bronze.

#### Mapeamento de Arquivos para Tabelas Bronze

| Arquivo Original | Nome da Tabela (Bronze) |
|---|---|
| olist_customers_dataset.csv | bronze.tb_customers |
| olist_geolocation_dataset.csv | bronze.tb_geolocalizacao |
| olist_order_items_dataset.csv | bronze.tb_order_items |
| olist_order_payments_dataset.csv | bronze.tb_order_payments |
| olist_order_reviews_dataset.csv | bronze.tb_order_reviews |
| olist_orders_dataset.csv | bronze.tb_orders |
| olist_products_dataset.csv | bronze.tb_products |
| olist_sellers_dataset.csv | bronze.tb_sellers |
| product_category_name_translation.csv | bronze.tb_product_category_name_translation |

### 3. Ingestão de API

Extrair a cotação do dólar via API do Banco Central e salvar na tabela `bronze.tb_cotacao_dolar`.

**Endpoint:**
```
https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json
```

- **Formato esperado de data:** `MM-DD-AAAA`
- Use **parâmetros do notebook** para as datas de início e fim.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS medalhao

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS landing;


In [0]:
%sql
USE CATALOG medalhao;

CREATE SCHEMA IF NOT EXISTS bronze;

USE SCHEMA bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

# Definição do caminho base no Unity Catalog
BASE_PATH = "/Volumes/medalhao/default/landing"

tabelas = {
    "olist_customers_dataset.csv":           "bronze.tb_customers",
    "olist_geolocation_dataset.csv":          "bronze.tb_geolocalizacao",
    "olist_order_items_dataset.csv":          "bronze.tb_order_items",
    "olist_order_payments_dataset.csv":       "bronze.tb_order_payments",
    "olist_order_reviews_dataset.csv":        "bronze.tb_order_reviews",
    "olist_orders_dataset.csv":               "bronze.tb_orders",
    "olist_products_dataset.csv":             "bronze.tb_products",
    "olist_sellers_dataset.csv":              "bronze.tb_sellers",
    "product_category_name_translation.csv":  "bronze.tb_product_category_name_translation",
}

# Iteração sobre o dicionário para processar cada arquivo individualmente
for arquivo, tabela in tabelas.items():
    
    # Concatenação do caminho completo do arquivo
    caminho = f"{BASE_PATH}/{arquivo}"

    # Leitura do arquivo CSV usando o Spark
    df = (
        spark.read
        .option("header", "true")      
        .option("inferSchema", "true")    
        .option("encoding", "UTF-8")    
        .csv(caminho)
    )

    # CONTROLE DE AUDITORIA
    df = df.withColumn("timestamp_ingestion", current_timestamp())

    # Escreve o DataFrame como uma tabela 
    (
        df.write
        .format("delta")                
        .mode("overwrite")               
        .option("overwriteSchema", "true") 
        .saveAsTable(tabela)             
    )

In [0]:
import requests
from pyspark.sql.functions import current_timestamp, to_timestamp

# Cria campos de WIDGETS 
dbutils.widgets.text("data_inicio_formatada", "01-01-2018")
dbutils.widgets.text("data_fim_formatada", "01-31-2026")

# Recupera os valores preenchidos nos widgets
data_inicio = dbutils.widgets.get("data_inicio_formatada")
data_fim = dbutils.widgets.get("data_fim_formatada")

# Utiliza parâmetros 'data_inicio' e 'data_fim' para filtrar datas, selecionar colunas e definir formato JSON
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'"
    f"&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra"
    f"&$format=json"
)

# Capturando os valores da API
response = requests.get(url)
dados = response.json().get("value", [])

# Verifica se a API retornou dados
if not dados:
    print(f"Nenhum dado retornado para o período: {data_inicio} até {data_fim}")
else:
    # Cria o DataFrame Spark a partir da lista de objetos Python
    df = spark.createDataFrame(dados)
    
    # Converte a coluna de data (String) para o tipo Timestamp
    df = df.withColumn("dataHoraCotacao", to_timestamp("dataHoraCotacao"))
    
    # CONTROLE DE AUDITORIA
    df = df.withColumn("timestamp_ingestion", current_timestamp())

    # Escrevendo na Bronze
    (
        df.write
        .format("delta")                 
        .mode("overwrite")               
        .option("overwriteSchema", "true") 
        .saveAsTable("bronze.tb_cotacao_dolar")
    )